In [1]:
import os
import pandas as pd
import numpy as np
from tqdm import tqdm

try:
    import catboost as cb
except:
    ! pip install catboost
    import catboost as cb

#### Functions

In [2]:
def get_catboost_feat_importance(cls_model_inference, pool_eval):
    df_tmp = cls_model_inference.get_feature_importance(
        data=pool_eval,
        type='LossFunctionChange',
        prettified=True,
    )
    df_tmp.columns = ['feature', 'importance']
    df_tmp.sort_values(by='importance', ascending=False, inplace=True)
    return df_tmp

#### Constants

In [3]:
str_project = os.getcwd().split('/')[4].replace('_','-')
print(f'Project: {str_project}')

str_task = os.getcwd().split('/')[5]
print(f'Task: {str_task}')

str_dirname_output = './output'

str_target = 'has_inst_tag'

list_str_inst = [
    # from ben: 2025-02-21
    'CURRENT',
    'SELF',
    # key words
    'CHIME-STRIDE',
    'CHIMEFINAL',
    # from dustin: 2025-02-24
    'SELF FIN',
    'SELF/LEAD',
    'SELFINC/LEAD',
    'SBNASELFLNDR',
    'SBNA SELF',
    'CHIME',
    'CLEO',
    'CLEO AI',
    'VARO',
    'ATLAS',
    'ATLCAPBKSELF',
    'POSSIBLE',
    'POSSIBLE FIN',
    'KIKOFF',
    'SUPER.COM',
    'STEP',
    'STEP MOBILE',
    'BRIGHT',
    'BRIGHT BLDR',
    'FIG TECH INC',
    'SELF/RENT',
    'SELFBILLSE',
    'PROGRESSRES',
    'FLEX',
    'FLEXFINANCE',
]

Project: 20250221-credit-builder-analysis
Task: 04_credit_builder_feats


#### Output dir

In [4]:
try:
    os.mkdir(str_dirname_output)
except:
    pass

#### Import data

In [5]:
str_filename = 'df.gzip'
str_uri = f's3://20241112-simple-model-test/08_prep_data/{str_filename}'
df = pd.read_parquet(
    str_uri,
)
# sort
df.sort_values(by='request_datetime', ascending=True, inplace=True)
df

,accountid,request_datetime,response_model_name,file_key,bitdebtor,bitdebtor__app,dealerstate__app,strdealershiptrackertype__app,strname__app,bitdealertrack__app,...,ENG-franchise,ENG-has_codebtor,ENG-vehicle_age,ENG-payment_to_income,ENG-loan_to_value,ENG-bk,ENG-perfect_payment_hx,ENG-perfect_payment_hx_open,ENG-perfect_payment_hx_closed,ENG-bk_x_wtd_avg
0,5702434,2021-07-26 16:29:29.3903686,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Iowa,Franchise,Iowa,True,...,1,0,7,NaN,1.311220,1,0,0,0,0.616667
1,5714239,2021-07-26 16:39:34.1121025,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Utah,Franchise,Utah,True,...,1,0,5,NaN,1.432368,0,0,0,0,NaN
2,5713063,2021-07-26 16:48:39.3211104,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Illinois,Franchise,Illinois,True,...,1,0,4,NaN,1.587073,1,0,0,0,NaN
3,5713732,2021-07-27 09:02:35.3300974,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Michigan,Independent,Michigan,False,...,0,0,2,NaN,1.081881,0,1,0,1,0.000000
4,5715634,2021-07-27 09:18:12.2190097,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Arizona,Franchise,Arizona,False,...,1,0,4,NaN,1.371350,0,1,0,1,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94472,8420588,2024-11-26 06:16:16+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1,North Carolina,Independent,North Carolina,True,...,0,0,4,NaN,1.280957,0,0,0,0,0.000000
94473,8401043,2024-11-26 06:21:44+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1,Nevada,Franchise,Nevada,True,...,1,0,0,NaN,1.198869,1,0,0,0,0.367073
94474,8414683,2024-11-26 06:25:09+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1,Virginia,Franchise,Virginia,False,...,1,0,3,NaN,1.313231,0,0,0,0,NaN
94475,8359085,2024-11-26 06:32:28+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1,Alabama,Franchise,Alabama,True,...,1,0,3,NaN,0.991586,1,0,0,0,0.006585


#### Create target

In [6]:
df['list_institutions'] = df['str_institution__tu_pmthx'].apply(
    lambda x: eval(x.replace('nan','None')),
)
df['list_institutions'] = df['list_institutions'].apply(
    lambda x: [] if x is None else x,
)

list_str_col_new = []
for str_inst in tqdm(list_str_inst):
    str_col_new = f'{str_inst}_tag'
    df[str_col_new] = df['list_institutions'].apply(
        lambda x: 1 if str_inst in x else 0,
    )
    list_str_col_new.append(str_col_new)

df['sum'] = df[list_str_col_new].sum(axis=1)
df['has_inst_tag'] = df['sum'].apply(
    lambda x: 1 if x > 0 else 0,
)
flt_mn = df['has_inst_tag'].mean()
print(f'Proportion has tag: {flt_mn:0.4f}')
# show
#df

100%|██████████| 29/29 [00:01<00:00, 20.18it/s]

Proportion has tag: 0.2053


#### Mark train/test

In [7]:
df['row'] = list(range(0, df.shape[0]))
df['prop_row'] = df['row'] / df.shape[0]
df['data_set'] = df['prop_row'].apply(
    lambda x: 'train' if x <= 0.8 else 'test',
)
df.drop(['row','prop_row'], axis=1, inplace=True)

#### List cols model

In [8]:
# rm target
list_cols_model = [col for col in df.columns if col != str_target]
# rm non-numeric
list_cols_object = [col for col in list_cols_model if df[col].dtype not in ['int64','float64']]
list_cols_model = [col for col in df.columns if col not in list_cols_object]
# rm tags
list_cols_model = [col for col in list_cols_model if 'tag' not in col.lower()]
# rm flag
list_cols_model = [col for col in list_cols_model if 'flag' not in col.lower()]
# rm loss
list_cols_model = [col for col in list_cols_model if 'loss' not in col.lower()]
# rm cols to ignore
list_cols_ignore = [
    'data_set',
    'sum',
    'accountid',
    'bigdebtorid__app',
    'days_on_books',
    'bigdebtorid__ln',
    'bigaccountid__ln',
    'strzipcode__app',
    'bitdefault__app',
    'dti__app',
    'pti__app',
    'prop_row',
    'fltapproveddebttoincome__app',
    'fltapprovedapr_contract__app',
    'fltacquisitionfee__app',
    'row',
    'payment__app',
]
list_cols_model = [col for col in list_cols_model if col not in list_cols_ignore]

#### Catboost model

In [9]:
# get train and test
df_train = df[df['data_set'] == 'train'].copy()
df_test = df[df['data_set'] == 'test'].copy()

# pool data
pool_train = cb.Pool(
    df_train[list_cols_model].copy(), 
    df_train[str_target], 
)
# pool
pool_valid = cb.Pool(
    df_test[list_cols_model].copy(), 
    df_test[str_target], 
)
# init class
cls_model_inference = cb.CatBoostClassifier(
    task_type='CPU',
    nan_mode='Min',
    random_state=42,
    eval_metric='AUC',
    iterations=100,
    learning_rate=None,
    class_weights=None,
    depth=None,
)
# fit
cls_model_inference.fit(
    pool_train,
    eval_set=[pool_valid],
    verbose=True,
    use_best_model=True,
    early_stopping_rounds=10, 
)

Learning rate set to 0.251118
0:	test: 0.8320961	best: 0.8320961 (0)	total: 95.4ms	remaining: 9.44s
1:	test: 0.8631967	best: 0.8631967 (1)	total: 138ms	remaining: 6.75s
2:	test: 0.8673892	best: 0.8673892 (2)	total: 179ms	remaining: 5.78s
3:	test: 0.8774530	best: 0.8774530 (3)	total: 223ms	remaining: 5.36s
4:	test: 0.8872382	best: 0.8872382 (4)	total: 265ms	remaining: 5.03s
5:	test: 0.8951279	best: 0.8951279 (5)	total: 306ms	remaining: 4.79s
6:	test: 0.8978682	best: 0.8978682 (6)	total: 352ms	remaining: 4.68s
7:	test: 0.9043988	best: 0.9043988 (7)	total: 396ms	remaining: 4.55s
8:	test: 0.9044213	best: 0.9044213 (8)	total: 441ms	remaining: 4.46s
9:	test: 0.9052144	best: 0.9052144 (9)	total: 484ms	remaining: 4.36s
10:	test: 0.9076813	best: 0.9076813 (10)	total: 527ms	remaining: 4.26s
11:	test: 0.9094192	best: 0.9094192 (11)	total: 566ms	remaining: 4.15s
12:	test: 0.9098600	best: 0.9098600 (12)	total: 610ms	remaining: 4.08s
13:	test: 0.9114035	best: 0.9114035 (13)	total: 652ms	remaining: 4

#### Get feat importance

In [10]:
# get importance
df_tmp = get_catboost_feat_importance(
    cls_model_inference=cls_model_inference,
    pool_eval=pool_valid,
)

#### Map description

In [11]:
# get the data dict
df_data_dict = pd.read_csv('data_dictionary.csv')
dict_map = dict(zip(df_data_dict['feature_name'], df_data_dict['Description']))
df_tmp['description'] = df_tmp['feature'].map(dict_map)

# save
str_filename = 'df_feat_imp.csv'
str_local_path = f'{str_dirname_output}/{str_filename}'
df_tmp.to_csv(str_local_path, index=False)

# show
df_tmp

,feature,importance,description
0,g201a__tu,0.047056,Total open to buy of open trades verified in p...
1,g202a__tu,0.046828,Total open to buy of open trades verified in p...
2,agg401__tu,0.009668,Aggregate non-mortgage actual payment for month 1
3,trd__tu,0.009136,Number of trades
4,at30s__tu,0.008314,Percentage of open trades > 50% of credit line...
...,...,...,...
2639,rev326__tu,-0.003888,Average Total Open-to-Buy for Revolving accoun...
2640,re102s__tu,-0.004340,Average credit line of open revolving trades v...
2641,rev316__tu,-0.004620,Max Total Open-to-Buy for Revolving accounts o...
2642,paymnt07__tu,-0.004996,Ratio of actual to minimum payment for non-mor...


#### Pivot

In [12]:
dict_agg = {col: 'mean' for col in df_tmp['feature']}
df_tmp = df.groupby(by=str_target, as_index=False).agg(dict_agg)
dict_map = {
    0: 'No',
    1: 'Yes',
}
df_tmp[str_target] = df_tmp[str_target].map(dict_map)
df_tmp

,has_inst_tag,g201a__tu,g202a__tu,agg401__tu,trd__tu,at30s__tu,s043s__tu,bi20s__tu,at03s__tu,paymnt10__tu,...,rev319__tu,at28a__tu,bi09s__tu,fr28s__tu,bc03s__tu,rev326__tu,re102s__tu,rev316__tu,paymnt07__tu,rev317__tu
0,No,1111.498473,1136.729417,520.589708,12.679400,78.844259,1.903134,98.661719,3.808096,4.693961,...,2255.679028,39022.936641,0.405812,1591.416447,1.902057,1046.542635,1201.454551,1426.515380,2.444777,1572.174615
1,Yes,1835.005791,1861.210154,704.027129,15.777829,60.154504,1.268843,62.417808,4.795341,6.119892,...,1919.110120,28149.845777,0.822669,1241.409495,1.910659,1018.481951,1013.560430,1457.112833,2.678118,1543.586905


#### Transpose

In [13]:
df_tmp_t = df_tmp.set_index(str_target).T
# show
df_tmp_t

has_inst_tag,No,Yes
g201a__tu,1111.498473,1835.005791
g202a__tu,1136.729417,1861.210154
agg401__tu,520.589708,704.027129
trd__tu,12.679400,15.777829
at30s__tu,78.844259,60.154504
...,...,...
rev326__tu,1046.542635,1018.481951
re102s__tu,1201.454551,1013.560430
rev316__tu,1426.515380,1457.112833
paymnt07__tu,2.444777,2.678118


#### Map description

In [14]:
# get the data dict
df_data_dict = pd.read_csv('data_dictionary.csv')
dict_map = dict(zip(df_data_dict['feature_name'], df_data_dict['Description']))
df_tmp_t['tmp'] = df_tmp_t.index
df_tmp_t['description'] = df_tmp_t['tmp'].map(dict_map)
df_tmp_t.drop('tmp', axis=1, inplace=True)
# show
df_tmp_t

has_inst_tag,No,Yes,description
g201a__tu,1111.498473,1835.005791,Total open to buy of open trades verified in p...
g202a__tu,1136.729417,1861.210154,Total open to buy of open trades verified in p...
agg401__tu,520.589708,704.027129,Aggregate non-mortgage actual payment for month 1
trd__tu,12.679400,15.777829,Number of trades
at30s__tu,78.844259,60.154504,Percentage of open trades > 50% of credit line...
...,...,...,...
rev326__tu,1046.542635,1018.481951,Average Total Open-to-Buy for Revolving accoun...
re102s__tu,1201.454551,1013.560430,Average credit line of open revolving trades v...
rev316__tu,1426.515380,1457.112833,Max Total Open-to-Buy for Revolving accounts o...
paymnt07__tu,2.444777,2.678118,Ratio of actual to minimum payment for non-mor...


#### Save

In [15]:
str_filename = 'df_feat_imp.csv'
str_local_path = f'{str_dirname_output}/{str_filename}'
df_tmp_t.to_csv(str_local_path, index=True)